In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, countDistinct, min, max

# Crear la sesión de Spark
spark = SparkSession.builder.appName("UnpivotDynamic").getOrCreate()

# Datos de ejemplo
data = [
    ("Nike", "Colombia", 1, "2024-10-01", 'Acne', 'Imperfection', 'M'),
    ("Adidas", "México", 2, "2024-10-02", 'Acne', 'Imperfection', 'M'),
    ("Nike", "Colombia", 3, "2024-10-03", 'Acne', 'Imperfection', 'M'),
    ("Nike", "Colombia", 4, "2024-10-06", 'Acne', 'Imperfection', 'M'),
    ("Nike", "Colombia", 5, "2024-10-10", 'Acne', 'Imperfection', 'F'),
    ("Nike", "Colombia", 6, "2024-10-03", 'Acne', 'Imperfection', 'F')
]

# Crear el DataFrame
columns = ["marca", "pais", "id", "fecha", "concern", "concern_type", "gender"]
df = spark.createDataFrame(data, columns)

# Verificar el esquema del DataFrame
#df.printSchema()

# 1. Identificar dinámicamente las columnas de atributos
columnas_atributos = [c for c in df.columns if c not in {"marca", "pais", "id", "fecha"}]

# 2. Crear la expresión para el unpivot con stack()
num_atributos = len(columnas_atributos)
stack_expr = ", ".join(
    [f"'{col}', {col}" for col in columnas_atributos]
)

# 3. Realizar el unpivot usando `stack()`
df_unpivot = df.select(
    "marca", "pais", "id", "fecha",
    expr(f"stack({num_atributos}, {stack_expr}) as (atributo, valor)")
)

# Verificar que el unpivot se realizó correctamente
#display(df_unpivot)

# 4. Agrupar y agregar: contar IDs, obtener fechas mínimas y máximas
df_result = (
    df_unpivot
    .groupBy("marca", "pais", "atributo", "valor")
    .agg(
        countDistinct("id").alias("num_ids"),
        min("fecha").alias("fecha_min"),
        max("fecha").alias("fecha_max")
    )
)

# Mostrar el resultado final
display(df_result)



marca,pais,id,fecha,atributo,valor
Nike,Colombia,1,2024-10-01,concern,Acne
Nike,Colombia,1,2024-10-01,concern_type,Imperfection
Nike,Colombia,1,2024-10-01,gender,M
Adidas,México,2,2024-10-02,concern,Acne
Adidas,México,2,2024-10-02,concern_type,Imperfection
Adidas,México,2,2024-10-02,gender,M
Nike,Colombia,3,2024-10-03,concern,Acne
Nike,Colombia,3,2024-10-03,concern_type,Imperfection
Nike,Colombia,3,2024-10-03,gender,M
Nike,Colombia,4,2024-10-06,concern,Acne


marca,pais,atributo,valor,num_ids,fecha_min,fecha_max
Adidas,México,concern,Acne,1,2024-10-02,2024-10-02
Adidas,México,concern_type,Imperfection,1,2024-10-02,2024-10-02
Adidas,México,gender,M,1,2024-10-02,2024-10-02
Nike,Colombia,concern,Acne,5,2024-10-01,2024-10-10
Nike,Colombia,concern_type,Imperfection,5,2024-10-01,2024-10-10
Nike,Colombia,gender,F,2,2024-10-03,2024-10-10
Nike,Colombia,gender,M,3,2024-10-01,2024-10-06
